--------------------------------------------------
# **Create Czech text dataset for training**
-------------------------------------------------

In [1]:
!apt install git-lfs

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
git-lfs is already the newest version (3.0.2-1ubuntu0.3).
0 upgraded, 0 newly installed, 0 to remove and 138 not upgraded.


## ***Imports***

In [2]:
import os, sys
import zipfile
import subprocess
import io
import shutil
import json
import polars as pl
import numpy as np
import pyarrow as pa
import pickle

import pyarrow.parquet as pq

from huggingface_hub import (
    get_full_repo_name,
    login,
    upload_folder,
    hf_hub_download,
    HfApi
)


from datasets import load_dataset
from kaggle_secrets import UserSecretsClient
from tqdm import tqdm

## ***Hugging Face Login***

In [3]:
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
user_email = user_secrets.get_secret("user_email")
user_name = user_secrets.get_secret("user_name")

login(token=hf_token)

### ***Data files***

1. ***CZLC/cermat_czech_mc***
   
   The **Cermat Czech MultiChoice** dataset was collected from assignments official CERMAT website. The dataset was collected from three tiers of assignments: 6 year, 9 year primary school test and final high school tests (so-called maturita). The assignments were semi-manually extracted from official PDFs available at [CERMAT's website](https://maturita.cermat.cz/menu/testy-a-zadani-z-predchozich-obdobi/matematika/testy-a-zadani-matematika).

    **Collection Date Range:** years 2016-2023

    ***Licensing and Credits***

    The majority of collection work was done by our student co-worker Jan Kapsa. Members of CZLC do not own the test assignments, neither are responsible for their contents.

    **Source:** https://huggingface.co/datasets/CZLC/cermat_czech_mc
 
2. ***hynky/czech-justice-summ-alpaca-long***

    **Source:** https://huggingface.co/datasets/hynky/czech-justice-summ-alpaca-long

3. ***hynky/czech_news_dataset_v2***

    Dataset containing the news articles from major online news outlets collected from 2000-2022.
    Follow-up paper https://arxiv.org/abs/2307.10666 (v1 of the dataset)

    Changes from v1:
        - Better contribution of novinky.cz in later stages
        - More articles, as a mistake in filtering was fixed.

   Collection was done using CmonCrawl.

   The dataset should be used for Research only purposes as I don't have rights for articles itself.

   **Source:** https://huggingface.co/datasets/hynky/czech_news_dataset_v2

4. ***davidadamczyk/czechbench_agree***

   This is an adapted and filtered test subset from the original Czech grammar agreement dataset

   **Source:** https://huggingface.co/datasets/davidadamczyk/czechbench_agree

6. ***davidadamczyk/czechbench_czech_news***

   This is a simplified and subsampled test subset from the original [czech_news_dataset_v2](https://huggingface.co/datasets/hynky/czech_news_dataset_v2).

   **Source:** https://huggingface.co/datasets/davidadamczyk/czechbench_czech_news

8. ***CIIRC-NLP/czech_news_simple-cs***

   **Source:** https://huggingface.co/datasets/CIIRC-NLP/czech_news_simple-cs

   This is a simplified and subsampled test subset from the original czech_news_dataset_v2.

   Only 5 basic news categories are considered:

   - Foreign
   - Local
   - Sports
   - Economy

   The test set includes 200 examples per category, 1000 examples in total. Apart from the category label, each example also contains the article's headline, brief summary, full textual content, optional keywords, original category specification, and URL.

   This dataset was created for use within the [Czech-Bench](https://gitlab.com/jirkoada/czech-bench) evaluation framework. 

In [4]:
ds01 = load_dataset("CZLC/cermat_czech_mc")
ds02 = load_dataset("hynky/czech-justice-summ-alpaca-long")
ds03 = load_dataset("hynky/czech_news_dataset_v2")
ds04 = load_dataset("davidadamczyk/czechbench_agree")
ds05 = load_dataset("davidadamczyk/czechbench_czech_news")
ds06 = load_dataset("CIIRC-NLP/czech_news_simple-cs", "default")

README.md: 0.00B [00:00, ?B/s]

test.jsonl: 0.00B [00:00, ?B/s]

Generating test split:   0%|          | 0/649 [00:00<?, ? examples/s]

README.md:   0%|          | 0.00/503 [00:00<?, ?B/s]

data/train-00000-of-00001-93f855e5cee8b5(…):   0%|          | 0.00/12.6M [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/20 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/980 [00:00<?, ? examples/s]

README.md: 0.00B [00:00, ?B/s]

data/test-00000-of-00001-8fd5c2a953da2a2(…):   0%|          | 0.00/2.67M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [5]:
print(type(ds01),'\n',ds01)
print(type(ds02),'\n',ds02)
print(type(ds03),'\n',ds03)
print(type(ds04),'\n',ds04)
print(type(ds05),'\n',ds05)
print(type(ds06),'\n',ds06)

<class 'datasets.dataset_dict.DatasetDict'> 
 DatasetDict({
    test: Dataset({
        features: ['type', 'query', 'choices', 'gold', 'context', 'subject', 'difficulty', 'source'],
        num_rows: 649
    })
})
<class 'datasets.dataset_dict.DatasetDict'> 
 DatasetDict({
    train: Dataset({
        features: ['output', 'instruction'],
        num_rows: 4560
    })
})
<class 'datasets.dataset_dict.DatasetDict'> 
 DatasetDict({
    train: Dataset({
        features: ['url', 'authors', 'headline', 'brief', 'keywords', 'category', 'content', 'comments_num', 'server', 'category_unclean', 'authors_gender', 'authors_cum_gender', 'day_of_week', 'date'],
        num_rows: 1641471
    })
    validation: Dataset({
        features: ['url', 'authors', 'headline', 'brief', 'keywords', 'category', 'content', 'comments_num', 'server', 'category_unclean', 'authors_gender', 'authors_cum_gender', 'day_of_week', 'date'],
        num_rows: 144836
    })
    test: Dataset({
        features: ['url', 'au

In [6]:
df01 = pl.from_arrow(ds01["test"].data.table)
df01

type,query,choices,gold,context,subject,difficulty,source
str,str,list[str],i64,str,str,str,str
"""MC""","""Které z následujících tvrzení …","[""A. Slovo patřit je v textu užito ve významu být součástí určité skupiny a slovo vést je v textu užito ve významu mít něco na starosti."", ""B. Slovo patřit je v textu užito ve významu být součástí určité skupiny a slovo vést je v textu užito ve významu mít něco za následek."", … ""D. Slovo patřit je v textu užito ve významu být ve vlastnictví někoho a slovo vést je v textu užito ve významu mít něco za následek.""]",1,"""Zprvu prorežimní spisovatel, n…","""český jazyk a literatura""","""maturitní zkouška""","""jaro 2021"""
"""MC""","""Ve kterém z následujících úsek…","[""A. za kritiku režimu si nadějný prozaik vysloužil zákaz"", ""B. tento rodák z Brna vstoupil do literatury"", … ""D. rodnou zemi mohl navštívit až po sametové revoluci""]",1,"""Zprvu prorežimní spisovatel, n…","""český jazyk a literatura""","""maturitní zkouška""","""jaro 2021"""
"""MC""","""Které z následujících tvrzení …","[""A. Zájmeno mu zastupuje velvyslance, zájmeno jeho přivlastňuje byt spisovateli."", ""B. Zájmeno mu zastupuje spisovatele, zájmeno jeho přivlastňuje byt velvyslanci."", … ""D. Zájmeno mu zastupuje spisovatele, zájmeno jeho přivlastňuje byt spisovateli.""]",3,"""Zprvu prorežimní spisovatel, n…","""český jazyk a literatura""","""maturitní zkouška""","""jaro 2021"""
"""MC""","""Které z následujících tvrzení …","[""A. Ve výchozím textu je chyba v koncovce přídavného jména."", ""B. Ve výchozím textu je chyba ve shodě přísudku s podmětem."", … ""D. Ve výchozím textu je chyba v psaní předpon s/z.""]",3,"""Hloučky nedočkavých lidí postá…","""český jazyk a literatura""","""maturitní zkouška""","""jaro 2021"""
"""MC""","""Na každé ze dvou vynechaných m…","[""A. oponentů – imponují"", ""B. oponentů – konkurují"", … ""D. konsenzů – konkurují""]",1,"""<bold>(1)</bold> Když poslouch…","""český jazyk a literatura""","""maturitní zkouška""","""jaro 2021"""
…,…,…,…,…,…,…,…
"""MC""","""Které z následujících tvrzení …","[""A. Slovo bohatý je v textu užito ve významu vyskytující se ve velkém množství, hojný."", ""B. Slovo lehký je v textu užito ve významu takový, jenž má malou hmotnost."", … ""D. Slovo příslušný je v textu užito ve významu mající právo něco udělat, oprávněný.""]",2,"""(1) Díky nálezům nástrojů, zbr…","""český jazyk a literatura""","""šestiletá gymnázia""","""2. řádný termín 2021"""
"""MC""","""Které z následujících tvrzení …","[""A. Výchozí text popírá informaci, že žádné známé písmo není starší než písmo piktografické."", ""B. Výchozí text popírá informaci, že piktografické písmo je jednodušší než nejstarší známé písmo."", … ""D. Z výchozího textu vyplývá, že žádné známé písmo není starší než písmo piktografické.""]",3,"""(1) Díky nálezům nástrojů, zbr…","""český jazyk a literatura""","""šestiletá gymnázia""","""2. řádný termín 2021"""
"""MC""","""Která z následujících možností…","[""A. jednoduchých textů, včetně komplikovaných"", ""B. ani jednoduchých, ani těch komplikovanějších textů"", … ""D. nejen jednoduchých, ale i komplikovaných textů""]",2,"""(1) Díky nálezům nástrojů, zbr…","""český jazyk a literatura""","""šestiletá gymnázia""","""2. řádný termín 2021"""


In [7]:
df02 = pl.from_arrow(ds03["train"].data.table)
df02

url,authors,headline,brief,keywords,category,content,comments_num,server,category_unclean,authors_gender,authors_cum_gender,day_of_week,date
str,list[str],str,str,list[str],i64,str,i32,i64,str,list[i64],i64,i64,datetime[μs]
"""https://www.idnes.cz/zpravy/za…",[],"""Svět bouřlivě oslavoval přícho…","""Ohňostroji, veselicemi a s jás…","[""Svatba"", ""Výpověď z pracovního poměru"", … ""Pivo""]",1,"""Ohnivá vlna u Lincolnova památ…",null,2,"""Zahraniční""",[],0,6,2000-01-01 00:00:00
"""https://www.idnes.cz/fotbal/pr…",[],"""Program fotbalistů: běžky a dř…","""Značně neoblíbené obodobí, tvr…","[""Ac sparta praha"", ""Sk slavia praha"", … ""Lázně""]",3,"""Menšímu chirurgickému zákroku …",null,2,"""Fotbal""",[],0,1,2000-01-03 00:00:00
"""https://www.idnes.cz/fotbal/pr…",[],"""Fotbalisté začali přípravu na …","""Do 19. února, kdy začne jarní …","[""Ac sparta praha"", ""Sk slavia praha"", … ""Lázně""]",3,"""Slavia: konečně s Kozlem Ligov…",null,2,"""Fotbal""",[],0,1,2000-01-03 00:00:00
"""https://www.idnes.cz/zpravy/do…",[],"""Tlustý: ODS bude žádat rekonst…","""ODS bude na pátečním jednání s…","[""Miloš zeman"", ""Mirek topolánek"", … ""Český rozhlas""]",2,"""ODS slibuje překvapení: změnu …",null,2,"""Domácí""",[],0,2,2000-01-04 00:00:00
"""https://www.idnes.cz/fotbal/pr…",[],"""Na ligu se chystají nedávní ma…","""V nesmírně vyrovnané tabulce f…","[""Ac sparta praha"", ""Sk slavia praha"", … ""Zemětřesení""]",3,"""Ostrava: žádné změny Kádr je s…",null,2,"""Fotbal""",[],0,3,2000-01-05 00:00:00
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""https://automix.denik.cz/nova-…","[""Radek Pecák""]","""Slovenský Land Rover Defender …","""Britsko-indická automobilka La…","[""Land"", ""Rover"", … ""Teaser""]",10,"""Zatím je to jen jedna fotka ma…",null,5,"""AUTO""",[1],1,0,null
"""https://automix.denik.cz/nova-…","[""Daniel Fuglevič""]","""Rakouská firma nabízí ikonické…","""Většina elektromobilů na trhu …",[],10,"""O firmě Kreisel Electric jste …",null,5,"""AUTO""",[1],1,0,null
"""https://automix.denik.cz/nova-…","[""Radek Pecák""]","""Poslední ""německý"" Passat odta…","""Ještě letos na jaře sjede z vý…","[""Volkswagen"", ""Volkswagen"", … ""Variant""]",10,"""Doby, kdy se auta takzvané man…",null,5,"""AUTO""",[1],1,0,null


In [8]:
df03_train = pl.from_arrow(ds03["train"].data.table)
df03_train

url,authors,headline,brief,keywords,category,content,comments_num,server,category_unclean,authors_gender,authors_cum_gender,day_of_week,date
str,list[str],str,str,list[str],i64,str,i32,i64,str,list[i64],i64,i64,datetime[μs]
"""https://www.idnes.cz/zpravy/za…",[],"""Svět bouřlivě oslavoval přícho…","""Ohňostroji, veselicemi a s jás…","[""Svatba"", ""Výpověď z pracovního poměru"", … ""Pivo""]",1,"""Ohnivá vlna u Lincolnova památ…",null,2,"""Zahraniční""",[],0,6,2000-01-01 00:00:00
"""https://www.idnes.cz/fotbal/pr…",[],"""Program fotbalistů: běžky a dř…","""Značně neoblíbené obodobí, tvr…","[""Ac sparta praha"", ""Sk slavia praha"", … ""Lázně""]",3,"""Menšímu chirurgickému zákroku …",null,2,"""Fotbal""",[],0,1,2000-01-03 00:00:00
"""https://www.idnes.cz/fotbal/pr…",[],"""Fotbalisté začali přípravu na …","""Do 19. února, kdy začne jarní …","[""Ac sparta praha"", ""Sk slavia praha"", … ""Lázně""]",3,"""Slavia: konečně s Kozlem Ligov…",null,2,"""Fotbal""",[],0,1,2000-01-03 00:00:00
"""https://www.idnes.cz/zpravy/do…",[],"""Tlustý: ODS bude žádat rekonst…","""ODS bude na pátečním jednání s…","[""Miloš zeman"", ""Mirek topolánek"", … ""Český rozhlas""]",2,"""ODS slibuje překvapení: změnu …",null,2,"""Domácí""",[],0,2,2000-01-04 00:00:00
"""https://www.idnes.cz/fotbal/pr…",[],"""Na ligu se chystají nedávní ma…","""V nesmírně vyrovnané tabulce f…","[""Ac sparta praha"", ""Sk slavia praha"", … ""Zemětřesení""]",3,"""Ostrava: žádné změny Kádr je s…",null,2,"""Fotbal""",[],0,3,2000-01-05 00:00:00
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""https://automix.denik.cz/nova-…","[""Radek Pecák""]","""Slovenský Land Rover Defender …","""Britsko-indická automobilka La…","[""Land"", ""Rover"", … ""Teaser""]",10,"""Zatím je to jen jedna fotka ma…",null,5,"""AUTO""",[1],1,0,null
"""https://automix.denik.cz/nova-…","[""Daniel Fuglevič""]","""Rakouská firma nabízí ikonické…","""Většina elektromobilů na trhu …",[],10,"""O firmě Kreisel Electric jste …",null,5,"""AUTO""",[1],1,0,null
"""https://automix.denik.cz/nova-…","[""Radek Pecák""]","""Poslední ""německý"" Passat odta…","""Ještě letos na jaře sjede z vý…","[""Volkswagen"", ""Volkswagen"", … ""Variant""]",10,"""Doby, kdy se auta takzvané man…",null,5,"""AUTO""",[1],1,0,null


In [9]:
df03_validation = pl.from_arrow(ds03["validation"].data.table)
df03_validation

url,authors,headline,brief,keywords,category,content,comments_num,server,category_unclean,authors_gender,authors_cum_gender,day_of_week,date
str,list[str],str,str,list[str],i64,str,i32,i64,str,list[i64],i64,i64,datetime[μs]
"""https://www.idnes.cz/sport/haz…","[""Petr Bílek""]","""Házenkáře Lovosic má do play o…","""Do roka a do dne skončil u ház…","[""Ústecký kraj"", ""Extraliga mužů"", … ""Jan landa""]",0,"""„Po dvanácti měsících spoluprá…",3,2,"""Ostatní""",[1],1,3,2020-03-04 00:00:00
"""https://www.idnes.cz/kultura/l…",[],"""Francouzi skupují Mor od Camus…","""Ve francouzských knihkupectvíc…","[""Francie"", ""Ernest hemingway"", … ""Paříž""]",4,"""Příběh románu z roku 1947 se o…",3,2,"""Kultura""",[],0,3,2020-03-04 00:00:00
"""https://www.idnes.cz/praha/zpr…","[""Jakub Augusta""]","""Námraza zkomplikovala ranní do…","""Námraza na troleji zkomplikova…","[""České dráhy"", ""Mrač"", … ""Benešov""]",2,"""Do normálu se doprava vrátila …",22,2,"""Kraje""",[1],1,3,2020-03-04 00:00:00
"""https://www.idnes.cz/kultura/f…","[""Mirka Spáčilová""]","""Zemřel recitátor Miroslav Ková…","""Popularizátor poezie Miroslav …","[""Karel kryl"", ""Vladimír merta"", … ""Mirek kovářík""]",4,"""Kovářík objevil řadu básnickýc…",10,2,"""Kultura""",[2],2,3,2020-03-04 00:00:00
"""https://www.idnes.cz/onadnes/m…","[""Markéta Škaldová""]","""Jarní trendy v líčení, které s…","""Líčení je pro většinu žen témě…","[""Trend"", ""Šálek"", … ""Sephora""]",21,"""Barevná kompozice Jak už to ta…",0,2,"""Ona""",[2],2,3,2020-03-04 00:00:00
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""https://automix.denik.cz/ojeta…","[""Dominik Valášek""]","""Vybíráme z inzerce: Auta znače…","""Dnes se v rámci našich inzertn…",[],10,"""Kdo by neznal smutný příběh sv…",null,5,"""AUTO""",[1],1,0,null
"""http://www.aktualne.cz/mudr-ja…",[],"""MUDr. Ján Pilka z Aesthevita: …","""Plastické operace u mužů už dá…","[""Právě se stalo"", ""Komerční článek"", … ""Operace očních víček""]",0,"""„Už dávno jsou pryč doby, kdy …",null,3,null,[],0,0,null
"""https://automix.denik.cz/nova-…","[""Jiří Švamberk""]","""KIA uvolnila dvě skici nového …","""Podoby reálného vozu bychom se…","[""Novinka"", ""Kia"", … ""Signature""]",10,"""Automobilka KIA uvedla, že des…",null,5,"""AUTO""",[1],1,0,null


In [10]:
df03_test = pl.from_arrow(ds03["test"].data.table)
df03_test

url,authors,headline,brief,keywords,category,content,comments_num,server,category_unclean,authors_gender,authors_cum_gender,day_of_week,date
str,list[str],str,str,list[str],i64,str,i32,i64,str,list[i64],i64,i64,datetime[μs]
"""https://www.seznamzpravy.cz/cl…","[""Tomáš Svoboda""]","""Co v kampani vystřídá koblihy?…","""Koronavirus obrátil zájem veře…","[""Premiér"", ""Andrej babiš"", … ""Volby""]",2,"""Kontaktní kampaň, tisíce kobli…",null,1,"""Volby""",[1],1,2,2021-02-16 00:00:00
"""https://www.novinky.cz/kultura…","[""Štěpán Kučera""]","""Bojím se, že budu jen stroj. R…","""Digitální René Descartes se zr…",[],4,"""Jaký je podle vás rozdíl mezi …",null,4,"""Kultura""",[1],1,2,2021-02-16 00:00:00
"""https://www.irozhlas.cz/zpravy…","[""Jaroslav Hroch""]","""‚Aby nemuseli tlouct data dvak…","""V Česku je už přes týden vakcí…","[""Koronavirus"", ""Koronavirus česko"", … ""Praha 7""]",2,"""Stojím v ordinaci nedaleko Let…",null,6,"""Zprávy z domova""",[1],1,2,2021-02-16 00:00:00
"""https://www.denik.cz/ze_sveta/…","[""Jaroslav Krupka""]","""Nehoda vlaků v Marylandu: sráž…","""Hustě sněžilo. Byl únor a po p…","[""Nehoda"", ""Srážka"", … ""Hasiči""]",0,"""Silver Spring vlastně není sam…",null,5,"""ZPRÁVY""",[1],1,2,2021-02-16 00:00:00
"""https://magazin.aktualne.cz/ku…","[""Magdalena Čechlovská""]","""Co viděl inspektor pražské obr…","""""V Nové pinakotéce se vůbec ne…","[""Viktor barvitius"", ""Drážďany"", … ""Česko""]",24,"""Barvitius byl pověřen Společno…",null,3,"""Výtvarné umění""",[2],2,2,2021-02-16 00:00:00
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""https://automix.denik.cz/nova-…","[""Jiří Švamberk""]","""Hyundai představuje své první …","""Novinka se sportovním stupněm …","[""Tucson"", ""n"", … ""Nošovice""]",10,"""Po představení stupně výbavy N…",null,5,"""AUTO""",[1],1,0,null
"""https://automix.denik.cz/magaz…","[""Daniel Fuglevič""]","""Tohle Bugatti je jediné na svě…","""Koncept Bugatti Vision GT, kte…",[],10,"""Bugatti Vision GT bylo v roce …",null,5,"""AUTO""",[1],1,0,null
"""https://automix.denik.cz/zivot…","[""Radek Pecák""]","""Test účinnosti klimatizace: Za…","""Testovaný exemplář Renaultu Mé…",[],10,"""Asi se vám už v létě nejednou …",null,5,"""AUTO""",[1],1,0,null


In [11]:
df04_train = pl.from_arrow(ds04["train"].data.table)
df04_train

sentence,choices,answer_idx
str,list[str],i64
"""____ se na postupu a oficiální…","[""Domluvila"", ""Domluvilo"", … ""Domluvil""]",2
"""Pondělí ____ očekávání meteoro…","[""splnila"", ""splnilo"", … ""splnil""]",1
"""Strašně se mu ____, jak na ní …","[""líbila"", ""líbilo"", … ""líbil""]",1
"""Nějakých šedesát let ____ z vr…","[""kouzlila"", ""kouzlilo"", … ""kouzlil""]",2
"""____ jste se vy s nějakým poku…","[""Setkala"", ""Setkalo"", … ""Setkal""]",4
…,…,…
"""Na doporučení jsme vlastně ___…","[""získala"", ""získalo"", … ""získal""]",2
"""Okolnosti rozchodu (to, zda je…","[""měla"", ""mělo"", … ""měl""]",3
"""A spíš nedosáhl útěchy, až krv…","[""uvrhnula"", ""uvrhnulo"", … ""uvrhnul""]",4


In [12]:
df04_test = pl.from_arrow(ds04["test"].data.table)
df04_test

sentence,choices,answer_idx
str,list[str],i64
"""Souhlasím, já ____ + - to samé…","[""napsala"", ""napsalo"", … ""napsal""]",4
"""____ jako leklá ryba se dvěma …","[""Vypadala"", ""Vypadalo"", … ""Vypadal""]",4
"""Podle ní by podnikatel ____ na…","[""měla"", ""mělo"", … ""měl""]",4
"""Ikonostas pochází s první polo…","[""dochovala"", ""dochovalo"", … ""dochoval""]",3
"""Dobře by ____ snad jen balerín…","[""vypadala"", ""vypadalo"", … ""vypadal""]",3
…,…,…
"""To pro mě ____ největší zážite…","[""byla"", ""bylo"", … ""byl""]",4
"""____ jsem do nich Cimrmana, Ma…","[""Tlačila"", ""Tlačilo"", … ""Tlačil""]",4
"""Vesmír ____ být prostě velice …","[""mohla"", ""mohlo"", … ""mohl""]",4


In [13]:
df05_train = pl.from_arrow(ds05["train"].data.table)
df05_train

url,authors,headline,brief,keywords,category,content,comments_num,server,category_unclean,authors_gender,authors_cum_gender,day_of_week,date,__index_level_0__,__index_level_1__
str,list[str],str,str,list[str],i64,str,f64,i64,str,list[i64],i64,i64,datetime[μs],i64,i64
"""https://www.idnes.cz/kultura/l…","[""Monika Zavřelová""]","""Napsali satirický životopis Ji…","""Po dvou Továrnách na debilno v…","[""Miloš zeman"", ""Jiří ovčáček"", … ""Satira""]",4,"""Satiru rád zaháním do extrému,…",130.0,2,"""Kultura""",[2],2,4,2021-02-18 00:00:00,4,56
"""https://www.idnes.cz/fotbal/re…","[""Miloslav Novák""]","""Německo se ztrapnilo až na kos…","""Jak ostudné. Jak trapné. Jak h…","[""Severní makedonie"", ""Uli hoeness"", … ""Timo werner""]",3,"""Německo v kvalifikaci o postup…",53.0,2,"""Fotbal""",[1],1,4,2021-04-01 00:00:00,3,29
"""https://www.seznamzpravy.cz/cl…",[],"""Statisíce lidí bez domova a tr…","""Víkendové devastující zemětřes…","[""Haiti"", ""Zemětřesení"", … ""Tropická bouře""]",1,"""„Země je fyzicky i mentálně zn…",null,1,"""Svět""",[],0,4,2021-08-19 00:00:00,1,126
"""https://www.seznamzpravy.cz/cl…","[""Jolana Humpálová""]","""Češi letěli přes půl planety. …","""Necelý rok už trvá blokáda v p…",[],1,"""První aktivisté obsadili okolí…",null,1,"""Svět""",[2],2,7,2021-06-20 00:00:00,1,175
"""https://www.irozhlas.cz/zpravy…",[],"""Ministerstvo poslalo ředitelům…","""Ministerstvo školství poslalo …","[""Koronavirus"", ""Koronavirus česko"", … ""Učitelé""]",2,"""Výběr pracovníků s přednostním…",null,6,"""Zprávy z domova""",[],0,3,2021-02-24 00:00:00,2,180
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""https://zpravy.aktualne.cz/eko…","[""Kateřina Hovorková""]","""Hypotéka jako dědictví. Banky …","""Hypotéka na 40 i více let se m…","[""Hypoteční úvěr"", ""Úvěr"", … ""Česká bankovní asociace""]",5,"""„Vidíme, že mnohé naše klienty…",null,3,"""Ekonomika""",[2],2,5,2021-10-29 00:00:00,7,19
"""https://zpravy.aktualne.cz/eko…","[""Kateřina Hovorková""]","""Banky v Česku prochází revoluc…","""Po několika letech, kdy se o k…","[""Banka"", ""Česko"", … ""Equa bank""]",5,"""Rakouská bankovní skupina Raif…",null,3,"""Ekonomika""",[2],2,4,2021-04-29 00:00:00,7,141
"""https://www.seznamzpravy.cz/cl…","[""Martin Čaban""]","""Vizita: Jak pošťouchnout k očk…","""Britský vědecký časopis Nature…","[""Newsletter vizita"", ""Zdravotnictví"", … ""Covid-19""]",2,"""Protože na stránkách asi nejpr…",null,1,"""Domácí""",[1],1,3,2022-06-08 00:00:00,2,131


In [14]:
df05_test = pl.from_arrow(ds05["test"].data.table)
df05_test

url,authors,headline,brief,keywords,category,content,comments_num,server,category_unclean,authors_gender,authors_cum_gender,day_of_week,date,__index_level_0__,__index_level_1__
str,list[str],str,str,list[str],i64,str,f64,i64,str,list[i64],i64,i64,datetime[μs],i64,i64
"""https://www.idnes.cz/oh/tokio-…",[],"""Kaťuša místo hymny na OH v Tok…","""Když si po olympijském triumfu…","[""Mezinárodní sportovní arbitráž"", ""Národní hymna"", … ""Tokio""]",3,"""Rusové musí oželet státní hymn…",13.0,2,"""Tokio 2021""",[],0,6,2021-03-13 00:00:00,3,45
"""https://www.irozhlas.cz/ekonom…",[],"""Cena bitcoinu spadla pod 30 00…","""Cena bitcoinu výrazně klesla a…","[""Bitcoin"", ""Burza"", … ""Akcie""]",5,"""„Globální trhy zasáhl rozsáhlý…",null,6,"""Ekonomika""",[],0,2,2021-07-20 00:00:00,7,20
"""https://prachaticky.denik.cz/f…","[""Bohumil Ortman""]","""Obstojí v sobotu Dynamo proti …","""Fotbalisté Dynama v úvodním ko…",[],3,"""Po fantastickém triumfu nad Sp…",null,5,"""Fotbal""",[1],1,6,2021-11-27 00:00:00,3,132
"""https://www.idnes.cz/zpravy/za…",[],"""V evropských zemích se hromadí…","""Zemím Evropské unie se ve skla…","[""Francie"", ""Itálie"", … ""Paříž""]",1,"""Například Francie do pátku zuž…",778.0,2,"""Zahraničí""",[],0,6,2021-02-27 00:00:00,1,127
"""https://hradecky.denik.cz/kult…","[""Pavla Horynová""]","""Hradecká kulturní pátračka zpe…","""Královéhradecké kulturní organ…",[],4,"""Iniciátorem projektu je Divadl…",null,5,"""Kultura""",[2],2,3,2021-05-05 00:00:00,4,173
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""https://www.idnes.cz/ekonomika…","[""Martin Petříček""]","""Ve stopách Avastu. Český start…","""Jeden z nejhodnotnějších start…","[""Spojené státy americké"", ""Dolar"", … ""Dublin""]",5,"""Productboard vytváří software,…",3.0,2,"""Ekonomika""",[1],1,4,2021-04-22 00:00:00,7,50
"""https://www.seznamzpravy.cz/cl…",[],"""Památky v Olomouckém kraji se …","""Hrady a zámky v Olomouckém kra…","[""Česká republika"", ""Olomoucký kraj"", … ""Velikonoce""]",2,"""V letošní sezóně půjde podle k…",null,1,"""Regiony""",[],0,2,2022-03-29 00:00:00,2,70
"""https://olomoucky.denik.cz/hok…","[""Ivan Němeček""]","""Přerovští hokejisté pomáhali n…","""Rodinu jednoho z juniorů přero…","[""Hokej"", ""Morava"", … ""Okres hodonín""]",3,"""Je úterý, šest hodin ráno a od…",null,5,"""Hokej""",[1],1,3,2021-06-30 00:00:00,3,109


In [15]:
df06 = pl.from_arrow(ds06["test"].data.table)
df06

url,headline,brief,keywords,category,content,category_unclean
str,str,str,list[str],i64,str,str
"""https://www.irozhlas.cz/ekonom…","""Soud potvrdil platnost pohledá…","""Pohledávky za více než deset m…","[""Okd"", ""Citibank"", … ""Pohledávka""]",5,"""V roce 2019 ostravský soud roz…","""Ekonomika"""
"""https://www.novinky.cz/ekonomi…","""Lego chce dát vale plastovým o…","""Lego bude balit stavebnice do …",[],5,"""Na zavádění nového typu balení…","""Ekonomika"""
"""https://www.idnes.cz/ekonomika…","""Hypoteční mejdan končí, úvěry …","""Objem poskytnutých hypoték za …","[""Česká spořitelna"", ""Čnb - česká národní banka"", … ""Fincentrum""]",5,"""Nad touto hranicí byla průměrn…","""Ekonomika"""
"""https://www.novinky.cz/ekonomi…","""K Milostivému létu se přidaly …","""Možností, jak se v rámci tzv. …",[],5,"""Milostivé léto je novelou exek…","""Ekonomika"""
"""https://www.idnes.cz/zpravy/za…","""V evropských zemích se hromadí…","""Zemím Evropské unie se ve skla…","[""Francie"", ""Itálie"", … ""Paříž""]",1,"""Například Francie do pátku zuž…","""Zahraničí"""
…,…,…,…,…,…,…
"""https://www.idnes.cz/fotbal/pr…","""Fotbalový pracant? Sadílek se …","""Z role pracanta ve středu pole…","[""Liberecký kraj"", ""Michal sadílek""]",3,"""Sadílek je pod Ještědem od loň…","""Fotbal"""
"""https://magazin.aktualne.cz/ku…","""Zesnulý Islanďan Jóhannsson sk…","""Skoro jako by po obloze připlo…","[""Hudba"", ""Jóhann jóhannsson"", … ""Denis villeneuve""]",4,"""Jóhannssonovo experimentální s…","""Film"""
"""https://zpravy.aktualne.cz/eko…","""HN: Mall Group zvažuje vstup n…","""Společnost Mall Group, která p…","[""Burza"", ""Hospodářské noviny"", … ""Rockaway capital""]",5,"""Uvedení akcionáři uvažují o pr…","""Ekonomika"""
